In [27]:
import numpy as np
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset():
    (X_train, y_train), (X_test, y_test) = mnist.load_data()
    # Flatten the images
    X_train = X_train.reshape(X_train.shape[0], -1)
    X_test = X_test.reshape(X_test.shape[0], -1)
    return X_train, y_train, X_test, y_test

X_train, labels_train, X_test, labels_test = load_dataset()
X_train, labels_train = shuffle(X_train, labels_train)
X_train, X_val, y_train, y_val = train_test_split(X_train, labels_train, test_size=0.33, random_state=42)

D = 10000  # dimensions in random space
IMG_LEN = 28
NUM_SAMPLES = X_train.shape[0]    # X_train.shape[0]

# Create a random map to the high dimensional space
print("Generating random projection...")
proj = np.random.rand(D, IMG_LEN * IMG_LEN)
def get_scene(img, proj):
    return np.dot(proj, img)

# Transform the image vectors into the hypervectors
def get_scenes(images, proj):
    return np.dot(images[:NUM_SAMPLES, :], proj.T)

print("Projecting images to higher dim space...")
X_train = get_scenes(X_train, proj)

digit_vectors = np.zeros((10, D))
for i in range(NUM_SAMPLES):
    digit_vectors[y_train[i]] += X_train[i]
digit_vectors = np.array(digit_vectors)
digit_vectors

def classify(images, digit_vectors):
    similarities = cosine_similarity(images, digit_vectors)
    classifications = np.argmax(similarities, axis=1)
    return classifications

predictions = classify(X_train, digit_vectors)
acc = accuracy_score(y_train[:X_train.shape[0]], predictions)*100
print("Train Accuracy: ",acc)

X_test_proj = get_scenes(X_test, proj)
predictions_test = classify(X_test_proj, digit_vectors)
acc_test = accuracy_score(labels_test, predictions_test) * 100
print("Test Accuracy: ", acc_test)

Generating random projection...
Projecting images to higher dim space...
Train Accuracy:  81.4726368159204


In [29]:
import numpy as np
import time
import sys
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset():
    (X_train, y_train), (X_test, y_test) = mnist.load_data()
    # Flatten the images
    X_train = X_train.reshape(X_train.shape[0], -1)
    X_test = X_test.reshape(X_test.shape[0], -1)
    return X_train, y_train, X_test, y_test

# Load and shuffle dataset
X_train, labels_train, X_test, labels_test = load_dataset()
X_train, labels_train = shuffle(X_train, labels_train)
X_train, X_val, y_train, y_val = train_test_split(X_train, labels_train, test_size=0.33, random_state=42)

D = 10000  # dimensions in random space
IMG_LEN = 28
NUM_SAMPLES = X_train.shape[0]

# Create a random map to the high dimensional space
print("Generating random projection...")
proj = np.random.rand(D, IMG_LEN * IMG_LEN)

def get_scene(img, proj):
    return np.dot(proj, img)

# Transform the image vectors into the hypervectors
def get_scenes(images, proj):
    return np.dot(images[:NUM_SAMPLES, :], proj.T)

print("Projecting images to higher dim space...")

# Measure training time
start_train_time = time.time()

X_train_proj = get_scenes(X_train, proj)

digit_vectors = np.zeros((10, D))
for i in range(NUM_SAMPLES):
    digit_vectors[y_train[i]] += X_train_proj[i]
digit_vectors = np.array(digit_vectors)

end_train_time = time.time()
training_time = end_train_time - start_train_time

def classify(images, digit_vectors):
    similarities = cosine_similarity(images, digit_vectors)
    classifications = np.argmax(similarities, axis=1)
    return classifications

# Training accuracy
predictions_train = classify(X_train_proj, digit_vectors)
acc_train = accuracy_score(y_train[:X_train_proj.shape[0]], predictions_train) * 100
print("Train Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data
X_test_proj = get_scenes(X_test, proj)

# Classify test data
predictions_test = classify(X_test_proj, digit_vectors)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

acc_test = accuracy_score(labels_test, predictions_test) * 100
print("Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = proj.nbytes + digit_vectors.nbytes

print("Training Time: {:.4f} seconds".format(training_time))
print("Inference Time: {:.4f} seconds".format(inference_time))
print("Model Memory: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating random projection...
Projecting images to higher dim space...
Train Accuracy:  81.3731343283582
Test Accuracy:  82.11
Training Time: 10.6226 seconds
Inference Time: 3.4880 seconds
Model Memory: 60.5774 MB


In [1]:
import numpy as np
import time
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset():
    iris = load_iris()
    X, y = iris.data, iris.target
    return X, y

# Load and shuffle dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Set dimensions for hyperdimensional space
D = 10000
NUM_CLASSES = 3
NUM_SAMPLES = X_train.shape[0]

# Create a random projection matrix
print("Generating random projection...")
proj = np.random.randn(D, X_train.shape[1])

def project_data(data, proj):
    return np.dot(data, proj.T)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train, proj)

# Create class hypervectors by summing the projected vectors of each class
class_hypervectors = np.zeros((NUM_CLASSES, D))
for i in range(NUM_SAMPLES):
    class_hypervectors[y_train[i]] += X_train_proj[i]

# Normalize the class hypervectors
class_hypervectors = class_hypervectors / np.linalg.norm(class_hypervectors, axis=1, keepdims=True)

end_train_time = time.time()
training_time = end_train_time - start_train_time

def classify(images, class_hypervectors):
    similarities = cosine_similarity(images, class_hypervectors)
    classifications = np.argmax(similarities, axis=1)
    return classifications

# Training accuracy
predictions_train = classify(X_train_proj, class_hypervectors)
acc_train = accuracy_score(y_train, predictions_train) * 100
print("Convensional HDC + Cosine Similarity Train Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, proj)

# Classify test data
predictions_test = classify(X_test_proj, class_hypervectors)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

acc_test = accuracy_score(y_test, predictions_test) * 100
print("Convensional HDC + Cosine Similarity Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = proj.nbytes + class_hypervectors.nbytes

print("Convensional HDC + Cosine Similarity Training Time: {:.4f} seconds".format(training_time))
print("Convensional HDC + Cosine Similarity Training Inference Time: {:.4f} seconds".format(inference_time))
print("Convensional HDC + Cosine Similarity Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating random projection...
Convensional HDC + Cosine Similarity Train Accuracy:  98.0
Convensional HDC + Cosine Similarity Test Accuracy:  96.0
Convensional HDC + Cosine Similarity Training Time: 0.0000 seconds
Convensional HDC + Cosine Similarity Training Inference Time: 0.0080 seconds
Convensional HDC + Cosine Similarity Model Memory Required: 0.5341 MB


In [2]:
import numpy as np
import time
import sys
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset():
    iris = load_iris()
    X, y = iris.data, iris.target
    return X, y

# Load and shuffle dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Set dimensions for hyperdimensional space
D = 10000

# Create a random projection matrix
print("Generating random projection...")
proj = np.random.randn(D, X_train.shape[1])

def project_data(data, proj):
    return np.dot(data, proj.T)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train, proj)

# Train an SVM classifier on the projected data
svm_classifier = SVC(kernel='linear')
svm_classifier.fit(X_train_proj, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = svm_classifier.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train) * 100
print("Convensional HDC + SVM Train Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, proj)

# Classify test data using the trained SVM classifier
predictions_test = svm_classifier.predict(X_test_proj)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

acc_test = accuracy_score(y_test, predictions_test) * 100
print("Convensional HDC + SVM Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = proj.nbytes + sys.getsizeof(svm_classifier)

print("Convensional HDC + SVM Training Time: {:.4f} seconds".format(training_time))
print("Convensional HDC + SVM Inference Time: {:.4f} seconds".format(inference_time))
print("Convensional HDC + SVM Model Memory: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating random projection...
Convensional HDC + SVM Train Accuracy:  98.0
Convensional HDC + SVM Test Accuracy:  100.0
Convensional HDC + SVM Training Time: 0.0163 seconds
Convensional HDC + SVM Inference Time: 0.0084 seconds
Convensional HDC + SVM Model Memory: 0.3052 MB
